# 04 - Physical Validation and Model Selection

A model can score well on held out data and still be wrong in a way no metric
catches. Shaking must weaken as you move away from an earthquake. That is not
a preference, it is what attenuation means. Nothing in a loss function says
so, so a model is free to produce a map where intensity rises with distance,
or wanders up and down, as long as it fits the training data.

The original version of this project checked by generating shake maps and
looking at them. That has two problems: it does not scale past a handful of
candidates, and nobody else can repeat it. Here the same judgement is made
numerically.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bench as B
import models as M
import spatial as S
from clean import apply_weight_scheme, expand_to_weighted_labels
from features import MODEL_FEATURES

pd.set_option("display.width", 150)

In [ ]:
features = pd.read_csv("../data/processed/features.csv")
labels = apply_weight_scheme(
    expand_to_weighted_labels(features, feature_columns=MODEL_FEATURES + ["mmi_mean"]),
    scheme="sqrt",
)
labels["is_weekend"] = labels["is_weekend"].astype(int)
train, test = M.split_by_event(labels, test_size=0.2, verbose=False)

results = pd.read_csv("../data/processed/bench_results.csv")
shortlist = results[~results["status"].str.startswith("failed")].nsmallest(14, "cell_mae")

fitted = {}
for candidate in [c for c in B.build_registry() if c.name in set(shortlist["model"])]:
    try:
        candidate.estimator.fit(train[MODEL_FEATURES], train["mmi"],
                                **B._weight_kwargs(candidate.estimator, train["weight"]))
    except TypeError:
        candidate.estimator.fit(train[MODEL_FEATURES], train["mmi"])
    fitted[candidate.name] = candidate

print(f"refitted {len(fitted)} candidates for inspection")

## The test

An earthquake is held fixed and distance is swept outwards, averaged over
eight compass bearings. Averaging matters: a model given azimuth features can
fall in one direction and rise in another, and testing a single bearing would
report whichever one happened to be sampled.

Three numbers, because models fail in three different ways. Rank correlation
asks whether the trend is downward at all. The fraction of decreasing steps
asks whether it gets there smoothly or in a blotchy zigzag. Total drop asks
whether it varies enough to be worth putting on a map.

In [ ]:
checks = []
for name, candidate in fitted.items():
    summary = S.physical_plausibility(candidate.estimator, MODEL_FEATURES, kind=candidate.kind)
    verdict = S.plausibility_verdict(summary)
    checks.append({
        "model": name,
        "worst_spearman": round(summary["worst_spearman"], 3),
        "fraction_decreasing": round(summary["mean_fraction_decreasing"], 3),
        "total_drop": round(summary["mean_total_drop"], 2),
        "passes": verdict["passes"],
        "why_not": "; ".join(verdict["reasons"]),
    })

physical = (pd.DataFrame(checks)
            .merge(shortlist[["model", "cell_mae", "auc_mmi6plus"]], on="model")
            .sort_values("cell_mae")
            .reset_index(drop=True))

physical[["model", "cell_mae", "worst_spearman", "fraction_decreasing", "total_drop", "passes"]]

Four of fourteen pass, and the most accurate model is not among them.

### Why strict monotonicity is the wrong test

Requiring every single step to decrease sounds correct. It is not, and the
table shows why.

In [ ]:
physical[physical["model"].str.contains("Decision Tree")][
    ["model", "cell_mae", "worst_spearman", "total_drop", "passes", "why_not"]
]

The shallow decision trees are the only strictly monotonic candidates, and
they are monotonic because they are nearly flat: about 0.4 MMI of variation
across the entire country. A map from one would be almost a single colour.

So the test is on the shape of the decay rather than on every step of it: a
strong downward trend, few reversals, and enough range to be worth mapping.

## Seeing the difference

The profiles make the distinction clearer than the numbers.

In [ ]:
best_overall = physical.iloc[0]
best_passing = physical[physical["passes"]].iloc[0]
flattest = physical.loc[physical["total_drop"].idxmin()]

fig, ax = plt.subplots(figsize=(9, 5))
for row, style in [(best_passing, "-"), (best_overall, "--"), (flattest, ":")]:
    candidate = fitted[row["model"]]
    profile = S.radial_profile(candidate.estimator, MODEL_FEATURES,
                               magnitude=6.5, kind=candidate.kind)
    ax.plot(profile["epicentral_distance_km"], profile["predicted_mmi"], style,
            label=f"{row['model']} (Spearman {row['worst_spearman']})")

ax.set_xscale("log")
ax.set_xlabel("Distance from epicentre (km)")
ax.set_ylabel("Predicted MMI")
ax.set_title("Attenuation profiles for a magnitude 6.5 earthquake")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Shake maps

The same comparison as a person would actually read it, for the largest
earthquake in the catalogue.

In [ ]:
event = {"magnitude": 7.8, "depth_km": 15.0, "longitude": 173.02, "latitude": -42.69}
grid = S.nz_grid(spacing_km=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, row in zip(axes, [best_passing, best_overall]):
    candidate = fitted[row["model"]]
    mapped = S.shake_map(candidate.estimator, MODEL_FEATURES, event, grid=grid,
                         kind=candidate.kind)
    scatter = ax.scatter(mapped["longitude"], mapped["latitude"],
                         c=mapped["predicted_mmi"], s=6, cmap="YlOrRd",
                         vmin=3, vmax=7)
    ax.plot(event["longitude"], event["latitude"], "b*", markersize=16)
    ax.set_title(f"{row['model']}\nSpearman {row['worst_spearman']}, "
                 f"{'passes' if row['passes'] else 'rejected'}", fontsize=10)
    ax.set_xlabel("Longitude")
    plt.colorbar(scatter, ax=ax, label="Predicted MMI")
axes[0].set_ylabel("Latitude")
plt.tight_layout()
plt.show()

The blue star marks the epicentre. A physically sound model shows intensity
falling smoothly outward from it. A rejected model shows patchy structure that
does not correspond to distance from the source.

## Selection

The rule is: satisfy the physics first, then take the most accurate model that
does. Accuracy alone would select a model whose maps cannot be trusted.

In [ ]:
best_overall = physical.iloc[0]
selected = physical[physical["passes"]].nsmallest(1, "cell_mae").iloc[0]

print("SELECTED")
print(f"  {selected['model']}")
print(f"  cell MAE          {selected['cell_mae']:.4f}")
print(f"  worst Spearman    {selected['worst_spearman']}")
print(f"  total drop        {selected['total_drop']} MMI")
print()
print("REJECTED DESPITE BETTER ACCURACY")
print(f"  {best_overall['model']}")
print(f"  cell MAE          {best_overall['cell_mae']:.4f}")
print(f"  worst Spearman    {best_overall['worst_spearman']}  (needs {S.MINIMUM_SPEARMAN})")
print()
cost = selected["cell_mae"] - best_overall["cell_mae"]
print(f"Accuracy given up to get trustworthy maps: {cost:.4f} MAE "
      f"({100 * cost / best_overall['cell_mae']:.1f}%)")

## Summary

- Only four of the fourteen most accurate models produce physically possible
  attenuation.
- The most accurate model of all is rejected. Its predictions do not fall
  reliably with distance, so its maps would mislead.
- Selecting on physics first costs about 10% of accuracy and buys a model
  whose output can be acted on.
- Constraining tree depth improves physical behaviour: Hist Gradient Boosting
  scores -0.939 unconstrained and -0.987 at depth 4. The smoother function is
  the better physical model and barely worse at prediction.

Metrics alone would have chosen differently, and worse. That is the finding
this project exists to make, and unlike the original version of it, the
judgement here is reproducible rather than made by eye.

In [ ]:
physical.to_csv("../data/processed/physical_checks.csv", index=False)
print("saved data/processed/physical_checks.csv")